## 🔗 Follow Me

<p align="center">
  <a href="https://github.com/Ramakm">
    <img src="https://img.shields.io/badge/GitHub-Follow-black?style=for-the-badge&logo=github" />
  </a>
  &nbsp;
  <a href="https://www.instagram.com/techwith.ram/">
    <img src="https://img.shields.io/badge/Instagram-Follow-E4405F?style=for-the-badge&logo=instagram&logoColor=white" />
  </a>
  &nbsp;
  <a href="https://x.com/techwith_ram">
    <img src="https://img.shields.io/badge/Twitter-Follow-1DA1F2?style=for-the-badge&logo=twitter&logoColor=white" />
  </a>
</p>

# 05 · Training (QLoRA with Unsloth)

> **Run on:** Google Colab (Runtime → Change runtime type → **T4 GPU**, free tier works). Sections marked *CPU-only* run anywhere.

Upload the `data/` folder from notebook 04 (or just re-run notebook 04 in the same Colab session).

In [ ]:
!pip install -q unsloth trl peft accelerate bitsandbytes datasets

In [ ]:
import torch
assert torch.cuda.is_available(), "Switch the Colab runtime to a GPU"
print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

## 1. Hyper-parameters

In [ ]:
BASE_MODEL   = "unsloth/Llama-3.2-3B-Instruct"   # ungated mirror of meta-llama/Llama-3.2-3B-Instruct
MAX_SEQ_LEN  = 1024
LORA_R       = 16
LORA_ALPHA   = 32
LR           = 2e-4
EPOCHS       = 1          # raise to 2-3 for small datasets
BATCH        = 2
GRAD_ACCUM   = 8          # effective batch = 16
OUTPUT_DIR   = "outputs"
ADAPTER_DIR  = "llama32-3b-finetuned-lora"

## 2. Load the 4-bit base model

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    dtype=None,
)

## 3. Attach LoRA adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
model.print_trainable_parameters()

## 4. Load data & apply the Llama-3.1/3.2 chat template

In [ ]:
import os
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

assert os.path.exists("data/train.jsonl"), "Run notebook 04 first (or upload the data/ folder)"
tokenizer = get_chat_template(tokenizer, chat_template="llama-3.2")

ds = load_dataset("json", data_files={"train": "data/train.jsonl", "val": "data/val.jsonl"})

def fmt(batch):
    return {"text": [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False) for m in batch["messages"]]}

ds = ds.map(fmt, batched=True)
print(ds["train"][0]["text"][:600])

## 5. Trainer (loss only on assistant tokens)

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds["train"],
    eval_dataset=ds["val"].select(range(min(200, len(ds["val"])))),
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LEN,
        per_device_train_batch_size=BATCH,
        per_device_eval_batch_size=BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        weight_decay=0.01,
        optim="adamw_8bit",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=100,
        save_strategy="no",
        output_dir=OUTPUT_DIR,
        seed=42,
        report_to="none",
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
)

In [ ]:
# sanity check: the prompt part must be masked (-100)
sample = trainer.train_dataset[0]
print(tokenizer.decode([t for t in sample["labels"] if t != -100])[:300])

## 6. Train
On a T4, 6k examples × 1 epoch ≈ 45–90 min. Watch that training loss decreases smoothly and eval loss doesn't rise (overfitting).

In [ ]:
stats = trainer.train()
print(f"{stats.metrics['train_runtime']/60:.1f} min · final train loss {stats.metrics['train_loss']:.3f}")
print(f"peak GPU memory: {torch.cuda.max_memory_reserved()/1e9:.1f} GB")

In [ ]:
import matplotlib.pyplot as plt
log = trainer.state.log_history
tr = [(l["step"], l["loss"]) for l in log if "loss" in l]
ev = [(l["step"], l["eval_loss"]) for l in log if "eval_loss" in l]
plt.plot(*zip(*tr), label="train")
if ev: plt.plot(*zip(*ev), "o-", label="val")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.show()

## 7. Save

In [ ]:
model.save_pretrained(ADAPTER_DIR)       # LoRA adapter only (~100 MB)
tokenizer.save_pretrained(ADAPTER_DIR)
print("saved", ADAPTER_DIR)

# Optional: merged 16-bit model for vLLM / HF inference
# model.save_pretrained_merged("llama32-3b-merged", tokenizer, save_method="merged_16bit")

# Optional: GGUF for Ollama / llama.cpp
# model.save_pretrained_gguf("llama32-3b-gguf", tokenizer, quantization_method="q4_k_m")

# Optional: push to the Hub
# model.push_to_hub("your-username/llama32-3b-finetuned-lora", token="hf_...")

Next → `06_Evaluation.ipynb`